# Lecture 6: Maneuvers 1

Orbital Mechanics, Technical University of Munich, winter semester 2026/27.

Companion notebook to the lecture: the rocket equation, the impulsive approximation and single-impulse maneuvers. The two-impulse coplanar transfers (Hohmann, bi-elliptic) are taught at the start of Lecture 7: see the example problems of that lecture. Every computation uses the library `pyastrodyn_edu`; nothing is redefined here.

**Conventions.** Units are km, s, kg. Angles are in radians inside the library; this notebook converts to degrees only to print. A vector name ends with its basis: `r_N` holds components in the inertial basis N, `dv_H` components in the Hill basis H (radial, transverse, normal).

Several examples are the validation problems of `docs/problems.md`, so the numbers printed here can be compared with that document.

Copyright © 2026 Dr. Marcello Romano. MIT License; see the file `LICENSE` of the repository.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import pyastrodyn_edu as ad
from pyastrodyn_edu import validation

mu = ad.MU_EARTH      # km^3/s^2
R = ad.R_EARTH        # km
DEG = np.pi / 180.0

print("pyastrodyn_edu version", ad.__version__)

## 1. The rocket equation: mass ratio and propellant

A geostationary satellite of 3500 kg must provide a velocity change of 1.47 km/s with an engine of specific impulse 320 s (Example problem 6.1).

In [ ]:
m0, dv, isp = 3500.0, 1.47, 320.0

ve = ad.exhaust_velocity(isp, ad.G0)
print(f"exhaust velocity  {ve:.4f} km/s")
print(f"mass ratio m0/mf  {ad.mass_ratio(dv, ve):.4f}")
print(f"propellant        {ad.propellant_mass(dv, ve, m0):.1f} kg")

The propellant fraction grows quickly with the velocity change, and the specific impulse sets how quickly. The functions accept arrays, so a whole curve is one call.

In [ ]:
dv_range = np.linspace(0.0, 6.0, 200)

fig, ax = plt.subplots(figsize=(6.5, 4))
for isp_value in (220.0, 320.0, 450.0):
    fraction = ad.propellant_mass(dv_range, ad.exhaust_velocity(isp_value, ad.G0), 1.0)
    ax.plot(dv_range, fraction, label=f"Isp = {isp_value:.0f} s")
ax.set_xlabel("velocity change (km/s)")
ax.set_ylabel("propellant mass / initial mass")
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 2. An impulsive burn in Hill components

In the impulsive approximation the position does not change during the burn and the velocity jumps by the velocity change. The natural basis to describe a burn is the Hill basis of the orbit: radial, transverse, normal.

Take the orbit of Example problem 5.1 and apply three burns of 0.1 km/s, one along each Hill axis. The table shows which elements each one changes.

In [ ]:
a, e = 9500.0, 0.15
i, raan, argp, f = np.array([52.0, 40.0, 110.0, 75.0]) * DEG
r_N, v_N = ad.coe2rv(a, e, i, raan, argp, f, mu)

C_HN = ad.dcm_HN(r_N, v_N)
print("velocity in Hill components (km/s):", np.round(C_HN @ v_N, 4))

def show(label, coe):
    a, e, i, raan, argp, f = coe
    print(f"{label:<12} a = {a:9.2f} km   e = {e:.5f}   i = {i/DEG:7.3f}   "
          f"raan = {raan/DEG:7.3f}   argp = {argp/DEG:8.3f} deg")

show("before", ad.rv2coe(r_N, v_N, mu))
for label, dv_H in (("radial", [0.1, 0, 0]), ("transverse", [0, 0.1, 0]), ("normal", [0, 0, 0.1])):
    show(label, ad.rv2coe(r_N, ad.apply_impulse(r_N, v_N, dv_H), mu))

A radial or transverse burn leaves the orbital plane unchanged (`i` and `raan` stay the same); a normal burn turns the plane and leaves the semi-major axis almost unchanged. This is the mechanics behind the *Reboost* game.

## 3. Where to burn: the Oberth effect

The change of specific energy caused by a velocity change is $\Delta\varepsilon = \mathbf{v}\cdot\Delta\mathbf{v} + \tfrac12 \Delta v^2$: the same burn adds more energy where the spacecraft is faster.

A geostationary transfer orbit with perigee altitude 400 km and apogee altitude 35786 km, and a prograde burn of 0.1 km/s (Example problem 6.3).

In [ ]:
r_p, r_a, dv = R + 400.0, R + 35786.0, 0.1
a = 0.5 * (r_p + r_a)
e = (r_a - r_p) / (r_a + r_p)

for name, f_burn in (("perigee", 0.0), ("apogee", np.pi)):
    r_N, v_N = ad.coe2rv(a, e, 0.0, 0.0, 0.0, f_burn, mu)
    dv_N = ad.apply_impulse(r_N, v_N, [0.0, dv, 0.0]) - v_N
    gain = ad.energy_change(v_N, dv_N)
    print(f"{name:<8} speed {np.linalg.norm(v_N):7.4f} km/s   energy gain {gain:.5f} km^2/s^2")

In [ ]:
f_all = np.linspace(0.0, 2 * np.pi, 361)
r_all_N, v_all_N = ad.coe2rv(a, e, 0.0, 0.0, 0.0, f_all, mu)       # 361 states in one call
speed = np.linalg.norm(v_all_N, axis=1)
gain = ad.energy_change(v_all_N, dv * v_all_N / speed[:, None])     # burn along the velocity

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(f_all / DEG, gain)
ax.set_xlabel("true anomaly of the burn (deg)")
ax.set_ylabel("energy gain (km$^2$/s$^2$)")
ax.set_xticks(range(0, 361, 60))
ax.grid(alpha=0.3)
plt.show()

## 4. Raising the apogee with a burn at perigee

A spacecraft at the perigee of an orbit with perigee altitude 300 km and apogee altitude 800 km fires along its velocity for 0.1 km/s (Example problem 6.2). The perigee does not move: a tangential burn at an apse changes only the opposite apse.

In [ ]:
r_p, r_a = R + 300.0, R + 800.0
a, e = 0.5 * (r_p + r_a), (r_a - r_p) / (r_a + r_p)

r_N, v_N = ad.coe2rv(a, e, 0.0, 0.0, 0.0, 0.0, mu)                 # at perigee
v_new_N = ad.apply_impulse(r_N, v_N, [0.0, 0.1, 0.0])
a_new, e_new, *_ = ad.rv2coe(r_N, v_new_N, mu)

print(f"perigee altitude  {ad.periapsis_radius(a_new, e_new) - R:8.2f} km  (was 300)")
print(f"apogee altitude   {ad.apoapsis_radius(a_new, e_new) - R:8.2f} km  (was 800)")

# The inverse question: which burn puts the apogee at 2000 km altitude?
print(f"burn for an apogee at 2000 km: {ad.apse_change_dv(r_p, r_a, R + 2000.0, mu):.4f} km/s")

In [ ]:
fig, ax = ad.plot_orbit_plane(a, e, body_radius=R, label="before")
ad.plot_orbit_plane(a_new, e_new, ax=ax, label="after 0.1 km/s at perigee", linestyle="--")
ax.plot(r_p, 0.0, "ko", markersize=4)
ax.set_xlabel("km"); ax.set_ylabel("km")
ax.legend(loc="upper right")
plt.show()

## 5. A general single impulse: the velocity triangle

A single impulse can move a spacecraft from one orbit to another only at a point that belongs to both. From a circular orbit of radius 7000 km to a coplanar orbit with semi-major axis 8400 km and eccentricity 0.2 (validation problem L6-5).

In [ ]:
r1, a2, e2 = 7000.0, 8400.0, 0.2
print("the orbits intersect:", bool(ad.orbits_intersect(r1, 0.0, 0.0, a2, e2, 0.0)))

f2 = np.arccos((ad.semilatus_rectum(a2, e2) / r1 - 1.0) / e2)      # crossing point, outbound
v1 = ad.circular_speed(r1, mu)
v2 = ad.visviva_speed(r1, a2, mu)
gamma2 = ad.flight_path_angle(e2, f2)

print(f"true anomaly on the new orbit  {f2/DEG:7.3f} deg")
print(f"speeds                         {v1:.4f} and {v2:.4f} km/s")
print(f"flight-path angle, new orbit   {gamma2/DEG:7.3f} deg")
print(f"velocity change                {ad.single_impulse_dv(v1, v2, gamma2):.4f} km/s")

# The same velocity change from the two velocity vectors at the common point.
_, v1_N = ad.coe2rv(r1, 0.0, 0.0, 0.0, 0.0, f2, mu)
_, v2_N = ad.coe2rv(a2, e2, 0.0, 0.0, 0.0, f2, mu)
print(f"check with vectors             {np.linalg.norm(v2_N - v1_N):.4f} km/s")

## Check against the validation problems

The stored results of the validation problems of this lecture, recomputed here.

In [ ]:
stored = validation.expected()
for problem in validation.PROBLEMS:
    if problem.lecture == 6:
        results = validation.to_plain(problem.results())
        same = all(
            value == stored[problem.id][name] if isinstance(value, (str, bool))
            else np.allclose(value, stored[problem.id][name], rtol=1e-9)
            for name, value in results.items()
        )
        print(f"{problem.id}  {problem.title:<62} {'agrees' if same else 'DIFFERS'}")